# ICB hierarchy — 2D visualization

We project the **ICB Subsectors** (FTSE Russell taxonomy, used by Russell 3000) into 2D, preserving the hierarchy `Industry → Supersector → Sector → Subsector`.

Source: [`icb-structure-and-definitions.xlsx`](https://www.lseg.com/content/dam/ftse-russell/en_us/documents/other/icb-structure-and-definitions.xlsx) (sheet `Mappable`, ~173 subsectors with definitions).

Three approaches:

- **A. UMAP — unsupervised** on text embeddings (semantic only)
- **B. UMAP — supervised** with `y = Industry` (semantic + hierarchy)
- **C. MDS** on a tree-distance matrix (pure hierarchy)

All three produce a Plotly scatter with the same encoding: **color = Industry**, **hover = full ICB path + definition**.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import umap
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import MDS

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ICB_PATH = PROJECT_ROOT / "notebooks" / "input" / "icb-structure-and-definitions.xlsx"
OUTPUT_HTML = PROJECT_ROOT / "notebooks" / "output" / "icb_hierarchy.html"
THEME_DASHBOARD_HTML = PROJECT_ROOT / "notebooks" / "output" / "theme_dashboard.html"
OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
RANDOM_SEED = 42

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"ICB file: {ICB_PATH}")
print(f"Device:   {DEVICE}")

ICB file: /Users/federicocinus/Progetti - Local/ThematicTrading/repo/notebooks/input/icb-structure-and-definitions.xlsx
Device:   mps


## 1. Load ICB taxonomy

Read the `Mappable` sheet (header is on the **second** row, data starts on the third).

In [2]:
icb_raw = pd.read_excel(ICB_PATH, sheet_name="Mappable", header=1)
icb_raw = icb_raw.dropna(subset=["Subsector"]).reset_index(drop=True)

icb = pd.DataFrame(
    {
        "industry_code": icb_raw["Industry code"].astype("Int64"),
        "industry": icb_raw["Industry"].str.strip(),
        "supersector_code": icb_raw["Supersector code"].astype("Int64"),
        "supersector": icb_raw["Supersector"].str.strip(),
        "sector_code": icb_raw["Sector code"].astype("Int64"),
        "sector": icb_raw["Sector"].str.strip(),
        "subsector_code": icb_raw["Subsector code"].astype("Int64"),
        "subsector": icb_raw["Subsector"].str.strip(),
        "definition": icb_raw["Definition"].fillna("").str.strip(),
    }
)
icb["path"] = (
    icb["industry"] + " › " + icb["supersector"] + " › " + icb["sector"] + " › " + icb["subsector"]
)

print(f"Subsectors:   {len(icb)}")
print(f"Sectors:      {icb['sector'].nunique()}")
print(f"Supersectors: {icb['supersector'].nunique()}")
print(f"Industries:   {icb['industry'].nunique()}")
icb.head()

Subsectors:   173
Sectors:      45
Supersectors: 20
Industries:   11


,industry_code,industry,supersector_code,supersector,sector_code,sector,subsector_code,subsector,definition,path
0,10,Technology,1010,Technology,101010,Software and Computer Services,10101010,Computer Services,Companies that provide consulting or integrati...,Technology › Technology › Software and Compute...
1,10,Technology,1010,Technology,101010,Software and Computer Services,10101015,Software,Publishers and distributors of computer softwa...,Technology › Technology › Software and Compute...
2,10,Technology,1010,Technology,101010,Software and Computer Services,10101020,Consumer Digital Services,Companies involved in digital platforms that p...,Technology › Technology › Software and Compute...
3,10,Technology,1010,Technology,101020,Technology Hardware and Equipment,10102010,Semiconductors,Producers and distributors of semiconductors a...,Technology › Technology › Technology Hardware ...
4,10,Technology,1010,Technology,101020,Technology Hardware and Equipment,10102015,Electronic Components,Companies involved in the application of high-...,Technology › Technology › Technology Hardware ...


## 2. Embed each Subsector

Concatenate `subsector + definition` and encode with the same finance embedder used in the news pipeline. One 768-d vector per subsector.

In [3]:
texts = (icb["subsector"] + ". " + icb["definition"]).tolist()

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Embedding matrix: {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix: (173, 768)


## 3. Shared Plotly helper

Each option (A, B, C) reuses this function so the look matches the BERTopic news plot.

In [4]:
INDUSTRY_PALETTE = px.colors.qualitative.Bold + px.colors.qualitative.Set3
INDUSTRY_ORDER = sorted(icb["industry"].unique())
INDUSTRY_COLORS = {ind: INDUSTRY_PALETTE[i % len(INDUSTRY_PALETTE)] for i, ind in enumerate(INDUSTRY_ORDER)}


def plot_icb(xy: np.ndarray, title: str, x_label: str = "x", y_label: str = "y"):
    plot_df = icb.assign(x=xy[:, 0], y=xy[:, 1])
    fig = px.scatter(
        plot_df,
        x="x",
        y="y",
        color="industry",
        category_orders={"industry": INDUSTRY_ORDER},
        color_discrete_map=INDUSTRY_COLORS,
        custom_data=["path", "definition"],
        title=title,
        labels={"x": x_label, "y": y_label, "industry": "Industry"},
        height=620,
    )
    fig.update_traces(
        marker={"size": 9, "line": {"width": 0.5, "color": "white"}},
        opacity=0.85,
        hovertemplate="<b>%{customdata[0]}</b><br><br>%{customdata[1]}<extra></extra>",
    )
    fig.update_layout(
        legend={"title": "Industry"},
        plot_bgcolor="#f8f9fa",
    )
    return fig

## Option A — UMAP unsupervised (semantic only)

UMAP on the raw FinLang embeddings. The layout reflects **textual similarity** between subsector definitions; the ICB hierarchy is not used as input.

In [5]:
umap_a = umap.UMAP(
    n_neighbors=10,
    min_dist=0.25,
    metric="cosine",
    random_state=RANDOM_SEED,
)
xy_a = umap_a.fit_transform(embeddings)

fig_a = plot_icb(xy_a, "A. UMAP — unsupervised (semantic only)", "UMAP 1", "UMAP 2")
fig_a.show()

## Step 3 — Theme dashboard (3 linked views)

Run BERTopic on a news window, then inspect themes with three complementary views:

1. **Assignment heatmap** — theme × ICB Sector (max cosine in 768-d space)
2. **Time heatmap** — theme × day (headline volume; persistence = horizontal stripes)
3. **Map** — UMAP on theme centroids; ICB Sectors as gray landmarks (no per-headline dots)

Bucket labels (cross-sector / sector story / orthogonal / nascent) come from the assignment matrix, not from the 2D layout.

In [6]:
import re
import polars as pl

NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "raw_news_2025.csv"
NEWS_DATE_START = "2025-01-01T00:00:00+00:00"
NEWS_DATE_END = "2025-01-08T00:00:00+00:00"  # first week
NEWS_SAMPLE = 8_000

MIN_TOPIC_SIZE = 80
MIN_SAMPLES = 15
TAU_COV = 0.22  # cosine threshold for "touched" sector
PERSIST_MIN_DOCS = 20  # min headlines/day to count as active

corpus = (
    pl.scan_csv(NEWS_PATH)
    .select(["Headline", "CaptureTime"])
    .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC"))
    .filter(
        (pl.col("CaptureTime") >= pl.lit(NEWS_DATE_START).str.to_datetime(time_zone="UTC"))
        & (pl.col("CaptureTime") < pl.lit(NEWS_DATE_END).str.to_datetime(time_zone="UTC"))
        & pl.col("Headline").is_not_null()
        & (pl.col("Headline").str.len_chars() > 0)
    )
    .unique(subset=["Headline"])
    .collect()
)
print(f"Unique headlines in window: {corpus.height:,}")
if corpus.height > NEWS_SAMPLE:
    corpus = corpus.sample(n=NEWS_SAMPLE, seed=RANDOM_SEED, shuffle=True)
print(f"Headlines used: {corpus.height:,}")

DATE_PATTERNS = (
    r"\b\d{4}[/\-]\d{1,2}[/\-]\d{1,2}\b",
    r"\b\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b",
    r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}\b",
    r"\bq[1-4]\s+\d{4}\b",
)


def strip_colon_prefix(text: str) -> str:
    if not text or ":" not in text:
        return text
    prefix = text.split(":", 1)[0].strip()
    if prefix and prefix[0].isalpha():
        return text.split(":", 1)[1].strip()
    return text


def strip_dates(text: str) -> str:
    for pattern in DATE_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    text = re.sub(r":\s*\d+\b", ":", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r":\s*$", "", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_headline(text: str) -> str:
    return strip_dates(strip_colon_prefix(text))


def preprocess(text: str) -> str:
    text = normalize_headline(text).lower()
    return " ".join(re.findall(r"[a-z]{3,}", text))


news_df = pd.DataFrame(
    {
        "Headline": corpus["Headline"].to_list(),
        "CaptureTime": corpus["CaptureTime"].to_list(),
    }
)
news_df["date"] = pd.to_datetime(news_df["CaptureTime"], utc=True).dt.date
news_headlines_raw = news_df["Headline"].tolist()
news_headlines = [normalize_headline(h) for h in news_headlines_raw]
clean_headlines = [preprocess(h) for h in news_headlines_raw]

keep = [i for i, t in enumerate(clean_headlines) if t.strip()]
news_df = news_df.iloc[keep].reset_index(drop=True)
news_headlines_raw = news_df["Headline"].tolist()
news_headlines = [news_headlines[i] for i in keep]
clean_headlines = [clean_headlines[i] for i in keep]

news_embeddings = embedder.encode(
    news_headlines,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"News embedding matrix: {news_embeddings.shape}")
print(f"Days in sample: {news_df['date'].nunique()}")

Unique headlines in window: 151,122
Headlines used: 8,000


Batches:   0%|          | 0/114 [00:00<?, ?it/s]

News embedding matrix: (7252, 768)
Days in sample: 7


In [7]:
bertopic_vectorizer = CountVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    min_df=2,
    max_df=0.9,
)
bertopic_umap = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_SEED,
)
bertopic_hdbscan = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE,
    min_samples=MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=bertopic_umap,
    hdbscan_model=bertopic_hdbscan,
    vectorizer_model=bertopic_vectorizer,
    verbose=False,
)
bertopic_topics, _ = topic_model.fit_transform(clean_headlines, news_embeddings)
news_df["topic"] = bertopic_topics

# --- ICB Sector centroids (45 sectors, mean of subsector embeddings) ---
sector_meta = (
    icb.groupby("sector", as_index=False)
    .agg(industry=("industry", "first"), n_subsectors=("subsector", "count"))
    .sort_values("sector")
    .reset_index(drop=True)
)
sector_embeddings = np.stack(
    [embeddings[icb["sector"] == s].mean(axis=0) for s in sector_meta["sector"]]
)
sector_embeddings /= np.linalg.norm(sector_embeddings, axis=1, keepdims=True)

# --- Theme centroids + labels ---
valid_topics = sorted(t for t in set(bertopic_topics) if t != -1)


def topic_label(topic_id: int, n_words: int = 4) -> str:
    words = topic_model.get_topic(topic_id) or []
    return ", ".join(w for w, _ in words[:n_words]) or f"topic {topic_id}"


theme_rows = []
for topic_id in valid_topics:
    mask = np.array(bertopic_topics) == topic_id
    centroid = news_embeddings[mask].mean(axis=0)
    centroid /= np.linalg.norm(centroid) + 1e-12
    theme_rows.append(
        {
            "topic": topic_id,
            "label": topic_label(topic_id),
            "n_docs": int(mask.sum()),
            "centroid": centroid,
        }
    )
def classify_theme(row) -> str:
    if row["n_sectors_touched"] >= 2 and row["ens"] >= 2:
        return "cross-sector"
    if row["n_sectors_touched"] == 1 and row["max_cos"] >= TAU_COV:
        return "sector story"
    if row["n_sectors_touched"] == 0 and row["ens"] >= 3:
        return "orthogonal"
    if row["max_cos"] < TAU_COV:
        return "nascent"
    return "other"


themes = pd.DataFrame(theme_rows).sort_values("n_docs", ascending=False).reset_index(drop=True)
theme_centroids = np.stack(themes["centroid"].tolist())

theme_sector_sim = theme_centroids @ sector_embeddings.T
themes["max_cos"] = theme_sector_sim.max(axis=1)
themes["novelty"] = 1 - themes["max_cos"]
themes["n_sectors_touched"] = (theme_sector_sim >= TAU_COV).sum(axis=1)
weights = np.clip(theme_sector_sim, 0, None)
weights = weights / (weights.sum(axis=1, keepdims=True) + 1e-12)
themes["ens"] = 1 / ((weights**2).sum(axis=1) + 1e-12)
themes["nearest_sector"] = sector_meta["sector"].to_numpy()[theme_sector_sim.argmax(axis=1)]
themes["bucket"] = themes.apply(classify_theme, axis=1)

dates = sorted(news_df["date"].unique())
time_matrix = (
    news_df[news_df["topic"] != -1]
    .groupby(["topic", "date"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=valid_topics, columns=dates, fill_value=0)
)
themes["active_days"] = (time_matrix.loc[themes["topic"]].values >= PERSIST_MIN_DOCS).sum(axis=1)
themes["total_days"] = len(dates)

print(f"BERTopic topics (excl. -1): {len(valid_topics)}")
print(f"Outliers: {(np.array(bertopic_topics) == -1).sum():,} ({(np.array(bertopic_topics) == -1).mean():.1%})")
print(themes[["topic", "label", "bucket", "n_docs", "active_days", "n_sectors_touched", "max_cos"]].head(12))

BERTopic topics (excl. -1): 20
Outliers: 2,102 (29.0%)
    topic                                label        bucket  n_docs  \
0       0      shares, acquires, million, sell  cross-sector     722   
1       1     sales, stocks, dollar, inflation  cross-sector     655   
2       2  conference, annual, meeting, report  cross-sector     401   
3       3          bowl, nfl, patriots, season    orthogonal     399   
4       4    year, winter, strategies, weather  cross-sector     374   
5       5         trump, court, investors, law  cross-sector     332   
6       6      ces, nvidia, microsoft, unveils  cross-sector     310   
7       7      limited, jan, industries, india  cross-sector     235   
8       8         bitcoin, crypto, xrp, solana  cross-sector     231   
9       9        golden, carter, awards, jimmy  cross-sector     211   
10     10       air, airlines, flight, flights  cross-sector     184   
11     11    appoints, chief, executive, names    orthogonal     175   

    acti

In [8]:
BUCKET_COLORS = {
    "cross-sector": "#e45756",
    "sector story": "#4c78a8",
    "orthogonal": "#72b7b2",
    "nascent": "#f58518",
    "other": "#b279a2",
}

theme_labels = [f"T{t} · {lbl}" for t, lbl in zip(themes["topic"], themes["label"])]

# --- 1. Assignment heatmap (theme × sector) ---
fig_assign = go.Figure(
    data=go.Heatmap(
        z=theme_sector_sim,
        x=sector_meta["sector"],
        y=theme_labels,
        colorscale="Viridis",
        colorbar={"title": "cosine"},
        hovertemplate="theme: %{y}<br>sector: %{x}<br>cos=%{z:.3f}<extra></extra>",
    )
)
fig_assign.update_layout(
    title="1. Theme × Sector assignment (768-d cosine)",
    height=max(420, 28 * len(themes)),
    xaxis={"tickangle": -45, "title": "ICB Sector"},
    yaxis={"title": "Theme (BERTopic)"},
)

# --- 2. Time heatmap (theme × day) ---
time_z = np.log1p(time_matrix.loc[themes["topic"]].values)
fig_time = go.Figure(
    data=go.Heatmap(
        z=time_z,
        x=[str(d) for d in dates],
        y=theme_labels,
        colorscale="YlOrRd",
        colorbar={"title": "log(1+docs)"},
        hovertemplate="theme: %{y}<br>day: %{x}<br>docs=%{customdata}<extra></extra>",
        customdata=time_matrix.loc[themes["topic"]].values,
    )
)
fig_time.update_layout(
    title="2. Theme × Day volume (horizontal stripes = persistent themes)",
    height=max(420, 28 * len(themes)),
    xaxis={"title": "Day"},
    yaxis={"title": "Theme (BERTopic)"},
)

# --- 3. Map: theme centroids + sector landmarks ---
if len(themes) >= 2:
    map_umap = umap.UMAP(
        n_neighbors=min(15, len(themes) - 1),
        min_dist=0.1,
        metric="cosine",
        random_state=RANDOM_SEED,
    ).fit(theme_centroids)
    xy_themes = map_umap.embedding_
    xy_sectors = map_umap.transform(sector_embeddings)
else:
    xy_themes = np.zeros((len(themes), 2))
    xy_sectors = np.zeros((len(sector_embeddings), 2))

fig_map = go.Figure()
fig_map.add_trace(
    go.Scatter(
        x=xy_sectors[:, 0],
        y=xy_sectors[:, 1],
        mode="markers",
        name="ICB sectors",
        marker={"size": 6, "color": "#bbb", "opacity": 0.5},
        text=sector_meta["sector"],
        hovertemplate="<b>%{text}</b><extra></extra>",
    )
)
for bucket, sub in themes.groupby("bucket"):
    idx = sub.index.to_numpy()
    fig_map.add_trace(
        go.Scatter(
            x=xy_themes[idx, 0],
            y=xy_themes[idx, 1],
            mode="markers+text",
            name=bucket,
            text=sub["label"],
            textposition="top center",
            textfont={"size": 9},
            marker={
                "size": np.clip(sub["n_docs"] / 8, 12, 40),
                "color": BUCKET_COLORS.get(bucket, "#666"),
                "line": {"width": 1, "color": "#222"},
            },
            customdata=np.stack(
                [sub["label"], sub["nearest_sector"], sub["n_docs"], sub["active_days"]], axis=1
            ),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "bucket: "
                + bucket
                + "<br>nearest sector: %{customdata[1]}<br>"
                "docs: %{customdata[2]} · active days: %{customdata[3]}<extra></extra>"
            ),
        )
    )
fig_map.update_layout(
    title="3. Theme map — UMAP on centroids, gray dots = ICB sectors",
    legend={"title": "Bucket"},
    plot_bgcolor="#f8f9fa",
    height=720,
)

# --- Export single HTML ---
sections = [
    ("Assignment heatmap", fig_assign),
    ("Time heatmap", fig_time),
    ("Theme map", fig_map),
]
html_parts = [
    "<html><head><meta charset='utf-8'><title>Theme dashboard</title>"
    "<style>body{font-family:-apple-system,Segoe UI,sans-serif;max-width:1200px;margin:24px auto;padding:0 16px;}"
    "h1{margin-bottom:4px;} h2{margin-top:36px;border-bottom:1px solid #eee;padding-bottom:4px;}</style></head><body>",
    f"<h1>Theme dashboard</h1><p>{len(themes)} themes · {news_df.shape[0]:,} headlines · "
    f"{news_df['date'].nunique()} days · embedder: <code>{EMBEDDING_MODEL}</code></p>",
]
for i, (title, fig) in enumerate(sections):
    html_parts.append(f"<h2>{title}</h2>")
    html_parts.append(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False))
html_parts.append("</body></html>")
THEME_DASHBOARD_HTML.write_text("\n".join(html_parts), encoding="utf-8")
print(f"Wrote {THEME_DASHBOARD_HTML}")

fig_assign.show()
fig_time.show()
fig_map.show()

Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/repo/notebooks/output/theme_dashboard.html
